In [10]:
import json
import tiktoken

enc = tiktoken.get_encoding("gpt2")

# -------- 1. Extract vocab --------
vocab = [
    enc.decode_bytes([i]).decode("latin-1")
    for i in range(enc.n_vocab)
]

# -------- 2. Extract merges (only keys containing b"\x00") --------
merges = []

for merge_token, rank in sorted(enc._mergeable_ranks.items(), key=lambda x: x[1]):

    if b"\x00" not in merge_token:
        # Not a real BPE merge; skip it
        continue

    a, b = merge_token.split(b"\x00", 1)

    merges.append([
        a.decode("latin-1"),
        b.decode("latin-1")
    ])

# -------- 3. Special tokens --------
special_tokens = enc._special_tokens

# -------- 4. Output JSON --------
out = {
    "tokenizer_type": "gpt2_bpe",
    "vocab_size": len(vocab),
    "vocab": vocab,
    "merges": merges,
    "special_tokens": special_tokens,
}

with open("gpt2_tokenizer.json", "w", encoding="latin-1") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
